In [2]:
import torch
import torch.nn as nn

In [5]:
batch_size = 1
seq_len = 3
embed_dim = 8
num_heads = 2
head_dim = embed_dim // num_heads

In [7]:
# Simulate a Query matrix after projection
# Shape: [batch_size, seq_len, embed_dim] -> [1, 3, 8]
Q = torch.rand(batch_size, seq_len, embed_dim)
print("Original Q shape:", Q.shape)

Original Q shape: torch.Size([1, 3, 8])


In [8]:
# Step 1: Reshape to separate out the heads
# New shape goal: [batch_size, seq_len, num_heads, head_dim]
Q = Q.view(batch_size, seq_len, num_heads, head_dim)
print("After splitting heads:", Q.shape)

After splitting heads: torch.Size([1, 3, 2, 4])


In [ ]:
# Step 2: Swap dimensions to put 'num_heads' before 'seq_len'
# Final shape goal: [batch_size, num_heads, seq_len, head_dim]
# This is required so that PyTorch operates on the last two dimensions (seq_len, head_dim)
Q = Q.transpose(1, 2)
print("Final Multi-Head Shape for Attention calculation:", Q.shape)

# Exercise

In [10]:
import torch

def split_heads(X, num_heads):
    batch_size, seq_len, embed_dim = X.shape
    head_dim = embed_dim // num_heads
    
    # 1. Split embed_dim into (num_heads, head_dim)
    # Target shape after this step: [batch_size, seq_len, num_heads, head_dim]
    X = X.view(batch_size, seq_len, num_heads, head_dim)
    
    # 2. Swap dimensions so num_heads comes before seq_len
    # Target shape after this step: [batch_size, num_heads, seq_len, head_dim]
    X = X.transpose(1,2)
    
    return X

def combine_heads(X):
    batch_size, num_heads, seq_len, head_dim = X.shape
    embed_dim = num_heads * head_dim
    
    # 1. Swap dimensions back to bring seq_len before num_heads
    # Target shape after this step: [batch_size, seq_len, num_heads, head_dim]
    # Hint: Use .transpose() or .permute()
    X = X.transpose(1,2)
    
    # 2. Flatten/Merge num_heads and head_dim back into embed_dim
    # Target shape after this step: [batch_size, seq_len, embed_dim]
    # Hint: Use .contiguous().view() to ensure memory layout allows reshaping
    X = X.contiguous().view(batch_size, seq_len, embed_dim)
    
    return X

# --- Test Data ---
B, SeqLen, EmbedDim, Heads = 2, 5, 8, 2

# Create a dummy tensor representing a projected Q, K, or V matrix
original_tensor = torch.randn(B, SeqLen, EmbedDim)
print("0. Starting shape:   ", original_tensor.shape)

# Transform to multi-head
split_tensor = split_heads(original_tensor, Heads)
print("1. After split_heads:", split_tensor.shape)
assert split_tensor.shape == (B, Heads, SeqLen, EmbedDim // Heads), "split_heads shape is incorrect!"

# Reconstruct back to original
reconstructed_tensor = combine_heads(split_tensor)
print("2. Reconstructed:    ", reconstructed_tensor.shape)
assert reconstructed_tensor.shape == (B, SeqLen, EmbedDim), "combine_heads shape is incorrect!"

0. Starting shape:    torch.Size([2, 5, 8])
1. After split_heads: torch.Size([2, 2, 5, 4])
2. Reconstructed:     torch.Size([2, 5, 8])
